In [7]:
import pyautogui
import time
import random

def human_scroll_and_download(total_pages):
    print("⚠️ You have 5 seconds to focus your Firefox window!")
    print("Make sure you are already on the FIRST page of your search.")
    time.sleep(5)

    for page in range(1, total_pages + 1):
        print(f"\n--- Processing Page {page} of {total_pages} ---")
        
        # 1. Wait for the page to fully load
        time.sleep(random.uniform(2.0, 5.0)) 
        
        # 2. Open the "Save Page As" dialog (Ctrl+S)
        print("Saving HTML...")
        pyautogui.hotkey('ctrl', 's')
        time.sleep(1) 

        # Type the filename and save
        filename = f"milano_stanze_page_{page}.html"
        pyautogui.write(filename)
        time.sleep(0.5)
        pyautogui.press('enter')

        # Wait a moment for the download to finish
        time.sleep(1)

        # If we just saved the last page, we don't need to click "Next"
        if page == total_pages:
            print("\nFinished downloading all pages!")
            break

        # 3. Human-like scroll to the bottom
        print("Scrolling down the page...")
        
        # We scroll down in 30-40 random bursts. 
        # (You may need to increase the range from 40 to 60 if the page is very long)
        scroll_steps = random.randint(35, 45)
        for _ in range(scroll_steps):
            # PyAutoGUI scroll amounts vary by OS; -200 is generally a smooth downward tick
            pyautogui.scroll(-350) 
            # Pause for a fraction of a second to mimic human reading speed
            time.sleep(random.uniform(0.1, 0.3))
            
        # Just to be 100% sure we hit the absolute bottom where the pagination is,
        # we press the 'End' key as a fail-safe.
        pyautogui.press('end')
        time.sleep(1.5)

        # 4. Click the Next Page button
        print(f"Clicking 'Next Page' at X:1269, Y:319...")
        
        # Move the mouse to the coordinates smoothly over 0.5 to 1.2 seconds
        pyautogui.moveTo(1269, 319, duration=random.uniform(0.5, 1.2), tween=pyautogui.easeInOutQuad)
        time.sleep(0.3) # Brief human pause before clicking
        pyautogui.click()
        
        # The loop will now restart and wait 6-8 seconds for the new page to load.

if __name__ == "__main__":
    PAGES_TO_SCRAPE = 52
    human_scroll_and_download(PAGES_TO_SCRAPE)

⚠️ You have 5 seconds to focus your Firefox window!
Make sure you are already on the FIRST page of your search.

--- Processing Page 1 of 52 ---
Saving HTML...
Scrolling down the page...
Clicking 'Next Page' at X:1269, Y:319...

--- Processing Page 2 of 52 ---
Saving HTML...
Scrolling down the page...
Clicking 'Next Page' at X:1269, Y:319...

--- Processing Page 3 of 52 ---
Saving HTML...
Scrolling down the page...
Clicking 'Next Page' at X:1269, Y:319...

--- Processing Page 4 of 52 ---
Saving HTML...
Scrolling down the page...
Clicking 'Next Page' at X:1269, Y:319...

--- Processing Page 5 of 52 ---
Saving HTML...
Scrolling down the page...
Clicking 'Next Page' at X:1269, Y:319...

--- Processing Page 6 of 52 ---
Saving HTML...
Scrolling down the page...
Clicking 'Next Page' at X:1269, Y:319...

--- Processing Page 7 of 52 ---
Saving HTML...
Scrolling down the page...
Clicking 'Next Page' at X:1269, Y:319...

--- Processing Page 8 of 52 ---
Saving HTML...
Scrolling down the page...
C

In [ ]:
import json
import glob
import os

def peek_at_json_structure(directory_path):
    file_list = glob.glob(os.path.join(directory_path, "*.html"))
    
    if not file_list:
        print("❌ No HTML files found in this directory.")
        return

    # We only need to look at the first file to understand the structure
    file_path = file_list[0]
    print(f"🔍 Opening {file_path} to peek at the JSON...")

    with open(file_path, 'r', encoding='utf-8') as file:
        html_content = file.read()

    try:
        json_string = html_content.split('<script id="__NEXT_DATA__" type="application/json">')[1].split('</script>')[0]
        data = json.loads(json_string)
        
        results = data['props']['pageProps']['dehydratedState']['queries'][0]['state']['data']['results']
        
        # Grab the VERY FIRST announcement in the list
        first_announcement = results[0]
        
        # 1. Print the top-level keys to the terminal so you can see them instantly
        print("\n--- Keys inside the 'realEstate' dictionary ---")
        for key in first_announcement.get('realEstate', {}).keys():
            print(f" - {key}")
            
        print("\n--- Keys inside the 'seo' dictionary ---")
        for key in first_announcement.get('seo', {}).keys():
            print(f" - {key}")

        # 2. Save the entire chunk to a readable file
        with open('sample_data.json', 'w', encoding='utf-8') as f:
            # indent=4 makes it pretty and readable instead of one giant block of text
            json.dump(first_announcement, f, indent=4, ensure_ascii=False)
            
        print("\n✅ I saved the full data structure to 'sample_data.json'.")
        print("Open that file in your text editor to see everything you can extract!")

    except Exception as e:
        print(f"⚠️ Error parsing JSON: {e}")

if __name__ == "__main__":
    HTML_FOLDER_PATH = "./full_data" 
    peek_at_json_structure(HTML_FOLDER_PATH)

🔍 Opening ./data\milano8stanze8page81.html to peek at the JSON...

--- Keys inside the 'realEstate' dictionary ---
 - visibility
 - dataType
 - id
 - uuid
 - advertiser
 - contract
 - isNew
 - luxury
 - price
 - properties
 - title
 - type
 - typology
 - hasMainProperty
 - isProjectLike
 - isMosaic

--- Keys inside the 'seo' dictionary ---
 - anchor
 - url

✅ I saved the full data structure to 'sample_data.json'.
Open that file in your text editor to see everything you can extract!


In [6]:
import json
import glob
import os
import csv

def extract_from_local_files(directory_path):
    all_announcements = []
    file_list = glob.glob(os.path.join(directory_path, "*.html"))
    
    print(f"🚀 Processing {len(file_list)} HTML files...")

    for file_path in file_list:
        # Read the file as a giant block of raw text
        with open(file_path, 'r', encoding='utf-8') as file:
            html_content = file.read()

        # 1. Slice the JSON out of the text directly (Lightning fast)
        try:
            json_string = html_content.split('<script id="__NEXT_DATA__" type="application/json">')[1].split('</script>')[0]
            data = json.loads(json_string)
        except (IndexError, json.JSONDecodeError):
            print(f"⚠️ Could not find or read JSON in: {file_path}")
            continue

        # 2. Extract the data using the corrected path
        try:
            results = data['props']['pageProps']['dehydratedState']['queries'][0]['state']['data']['results']
            
            for item in results:
                # 'realEstate' and 'seo' are separate dictionaries side-by-side
                real_estate = item.get('realEstate', {})
                seo = item.get('seo', {}) 
                
                url = seo.get('url')
                title = real_estate.get('title', 'N/A')
                
                # Dig into the price dictionary
                price_dict = real_estate.get('price', {})
                price = price_dict.get('formattedValue', 'N/A')
                
                # If we found a URL, save the room
                if url:
                    all_announcements.append({
                        'title': title,
                        'price': price,
                        'url': url
                    })
                    
        except KeyError as e:
            print(f"⚠️ Unexpected JSON structure in {file_path}. Missing: {e}")

    return all_announcements

def save_to_csv(data, output_folder="prices", filename="extracted_rooms.csv"):
    # Create the folder if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)
    filepath = os.path.join(output_folder, filename)
    
    # Save the file (utf-8-sig ensures Excel reads Euro € symbols correctly)
    with open(filepath, 'w', newline='', encoding='utf-8-sig') as file:
        writer = csv.DictWriter(file, fieldnames=['title', 'price', 'url'])
        writer.writeheader()
        writer.writerows(data)
        
    print(f"✅ Successfully saved {len(data)} unique rooms to '{filepath}'")

if __name__ == "__main__":
    # Point this to the folder containing your HTML files
    HTML_FOLDER_PATH = "./data" 
    
    extracted_data = extract_from_local_files(HTML_FOLDER_PATH)
    
    if extracted_data:
        save_to_csv(extracted_data)
    else:
        print("❌ Extraction failed. 0 rooms found.")

🚀 Processing 80 HTML files...
✅ Successfully saved 2000 unique rooms to 'prices\extracted_rooms.csv'


In [11]:
import json
import glob
import os
import csv

def extract_from_local_files(directory_path):
    all_announcements = []
    file_list = glob.glob(os.path.join(directory_path, "*.html"))
    
    print(f"🚀 Processing {len(file_list)} HTML files...")

    for file_path in file_list:
        with open(file_path, 'r', encoding='utf-8') as file:
            html_content = file.read()

        try:
            json_string = html_content.split('<script id="__NEXT_DATA__" type="application/json">')[1].split('</script>')[0]
            data = json.loads(json_string)
        except (IndexError, json.JSONDecodeError):
            print(f"⚠️ Could not find or read JSON in: {file_path}")
            continue

        try:
            results = data['props']['pageProps']['dehydratedState']['queries'][0]['state']['data']['results']
            
            for item in results:
                real_estate = item.get('realEstate', {})
                seo = item.get('seo', {}) 
                
                # --- LEVEL 1: Top Level Data ---
                ad_id = real_estate.get('id', '')
                title = real_estate.get('title', '')
                url = seo.get('url', '')
                contract = real_estate.get('contract', '')
                
                # --- LEVEL 2: Price ---
                price_dict = real_estate.get('price', {})
                price_value = price_dict.get('value', '')
                price_formatted = price_dict.get('formattedValue', '')
                
                # --- LEVEL 3: Agency Info ---
                advertiser = real_estate.get('advertiser', {})
                agency = advertiser.get('agency', {})
                agency_name = agency.get('displayName', 'Privato/Nessuna')
                
                # Extract the first phone number if it exists
                phones = agency.get('phones', [])
                phone_number = phones[0].get('value', '') if phones else ''
                
                # --- LEVEL 4: Properties (The core details) ---
                # Properties is usually a list with one dictionary inside
                properties_list = real_estate.get('properties', [{}])
                props = properties_list[0] if properties_list else {}
                
                rooms = props.get('rooms', '')
                surface = props.get('surface', '')
                bathrooms = props.get('bathrooms', '')
                heating = props.get('ga4Heating', '')
                
                floor_dict = props.get('floor', {})
                floor = floor_dict.get('value', '')
                
                # Clean the description
                raw_desc = props.get('description', '')
                clean_desc = raw_desc.replace('\n', ' ').replace('\r', '')
                
                # Join the features array into a single comma-separated string
                features_list = props.get('ga4features', [])
                features = ", ".join(features_list)
                
                # Count the photos
                photos = props.get('multimedia', {}).get('photos', [])
                photo_count = len(photos)
                
                # --- LEVEL 5: Location ---
                location = props.get('location', {})
                address = location.get('address', '')
                macrozone = location.get('macrozone', '')
                microzone = location.get('microzone', '')
                lat = location.get('latitude', '')
                lng = location.get('longitude', '')

                # Only append if we actually have a URL (valid listing)
                if url:
                    all_announcements.append({
                        'id': ad_id,
                        'title': title,
                        'price_value': price_value,
                        'price_formatted': price_formatted,
                        'rooms': rooms,
                        'bathrooms': bathrooms,
                        'surface': surface,
                        'floor': floor,
                        'heating': heating,
                        'features': features,
                        'address': address,
                        'macrozone': macrozone,
                        'microzone': microzone,
                        'latitude': lat,
                        'longitude': lng,
                        'agency_name': agency_name,
                        'phone_number': phone_number,
                        'photo_count': photo_count,
                        'url': url,
                        'description': clean_desc
                    })
                    
        except KeyError as e:
            print(f"⚠️ Unexpected JSON structure in {file_path}. Missing: {e}")

    return all_announcements

def save_to_csv(data, output_folder="prices", filename="extracted_rooms_4.csv"):
    os.makedirs(output_folder, exist_ok=True)
    filepath = os.path.join(output_folder, filename)
    
    # Grab the keys from the first dictionary to use as CSV headers
    if not data:
        return
    headers = list(data[0].keys())
    
    with open(filepath, 'w', newline='', encoding='utf-8-sig') as file:
        writer = csv.DictWriter(file, fieldnames=headers)
        writer.writeheader()
        writer.writerows(data)
        
    print(f"✅ Successfully saved {len(data)} unique rooms to '{filepath}'")

if __name__ == "__main__":
    HTML_FOLDER_PATH = "./full_data_3" 
    
    extracted_data = extract_from_local_files(HTML_FOLDER_PATH)
    
    if extracted_data:
        save_to_csv(extracted_data)
    else:
        print("❌ Extraction failed. 0 rooms found.")

🚀 Processing 44 HTML files...
✅ Successfully saved 1082 unique rooms to 'prices\extracted_rooms_4.csv'


In [14]:


import pandas as pd
import os

def clean_merge(file_list, output_filename="final_dataset.csv"):
    dfs = []
    
    for file in file_list:
        # 'utf-8-sig' handles the hidden BOM characters that cause 'Â'
        df = pd.read_csv(file, encoding='utf-8-sig')
        
        # Strip potential hidden whitespace from column names just in case
        df.columns = df.columns.str.strip()
        
        # Check against the first file's columns
        if not dfs or list(df.columns) == list(dfs[0].columns):
            print(f"Processing: {file}")
            dfs.append(df)
        else:
            print(f"Skipping {file}: Column mismatch.")

    if dfs:
        result = pd.concat(dfs, ignore_index=True)
        # Saving with utf-8-sig makes it look perfect in Excel too
        result.to_csv(output_filename, index=False, encoding='utf-8-sig')
        print(f"\nDone! Created {output_filename}")

# Example Usage:
files = ["./prices/extracted_rooms_1.csv","./prices/extracted_rooms_2.csv", "./prices/extracted_rooms_3.csv", "./prices/extracted_rooms_4.csv"]
clean_merge(files)

Processing: ./prices/extracted_rooms_1.csv
Processing: ./prices/extracted_rooms_2.csv
Processing: ./prices/extracted_rooms_3.csv
Processing: ./prices/extracted_rooms_4.csv

Done! Created final_dataset.csv
